In [1]:
import torch
import numpy as np

from src.nano_maker.skeleton import Skeleton
from src.nano_maker.container import configs

from src.nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

In [2]:
c = configs.skeleton_config
model = Skeleton(n_embd=c['n_embd'], n_head=c['n_head'], n_layers=c['n_layers'],
                      block_size=c['block_size'], map4_res=c['map4_res'], max_angstroms=c['max_angstroms'],dropout=c['dropout'])

In [5]:
from src.paths import *
sk_wt_path = PROJECT_ROOT / "src/nano_maker/container/skeleton_E3I3.pt"

In [6]:
prototype_weights = torch.load(sk_wt_path, map_location="cpu")

In [7]:
print(type(prototype_weights))

<class 'dict'>


In [8]:
model.load_state_dict(prototype_weights["model_state_dict"])

<All keys matched successfully>

In [9]:
device = 'cpu'
block_size = c["block_size"]
max_angstroms = c["max_angstroms"]

In [10]:
sample_smiles = "CCNC(=O)[C@H]1CCCN1C(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)[C@H](CO)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)CCc1ccc(Cl)cc1 |wU:57.63,12.20,wD:45.57,31.44,23.28,5.4,63.67,(30.68,-5.02,;30.68,-6.54,;29.36,-7.34,;28.03,-6.58,;26.73,-7.34,;28,-5.06,;29.2,-4.18,;28.69,-2.75,;27.18,-2.75,;26.75,-4.2,;25.41,-4.94,;25.41,-6.48,;24.09,-4.18,;24.09,-2.64,;25.44,-1.88,;25.44,-.33,;26.79,.42,;26.79,1.95,;28.13,2.73,;25.47,2.74,;22.76,-4.94,;21.41,-4.16,;21.44,-2.62,;20.08,-4.9,;20.06,-6.45,;21.4,-7.24,;21.38,-8.77,;22.73,-6.47,;18.74,-4.13,;17.42,-4.89,;17.41,-6.42,;16.1,-4.12,;16.1,-2.59,;15.64,-1.13,;14.17,-.64,;14.17,.9,;15.65,1.37,;16.26,2.77,;17.81,2.93,;18.71,1.7,;18.07,.28,;16.54,.12,;14.77,-4.87,;13.43,-4.1,;13.43,-2.55,;12.08,-4.86,;12.08,-6.38,;13.4,-7.18,;13.4,-8.72,;14.72,-9.49,;16.07,-8.73,;17.39,-9.51,;16.07,-7.19,;14.75,-6.42,;10.76,-4.07,;9.44,-4.83,;9.41,-6.38,;8.11,-4.06,;8.12,-2.52,;9.44,-1.77,;6.77,-4.81,;5.44,-4.03,;5.45,-2.51,;4.1,-4.8,;4.09,-6.34,;5.42,-7.12,;6.83,-6.5,;7.85,-7.64,;7.06,-8.98,;7.53,-10.44,;6.5,-11.57,;5,-11.24,;4.52,-9.78,;5.57,-8.66,;2.78,-4.03,;1.43,-4.77,;1.45,-6.31,;.08,-4.03,;-1.37,-4.99,;-2.89,-4.2,;-2.98,-2.49,;-4.53,-1.72,;-5.95,-2.65,;-7.46,-1.8,;-5.86,-4.38,;-4.33,-5.15,)|"

sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
sample_skeleton = model.generate(sample_map4)
sample_skeleton

tensor([[ 1.6568e+01,  1.7319e+00,  1.5057e+00],
        [ 1.5356e+01,  1.5184e+00,  1.5981e+00],
        [ 1.4992e+01,  1.6651e+00,  1.5025e+00],
        [ 1.4330e+01,  1.2205e+00,  1.2933e+00],
        [ 1.4026e+01,  1.3905e+00,  1.9733e+00],
        [ 1.3602e+01,  1.3844e+00,  9.9348e-01],
        [ 1.3414e+01,  2.0995e+00,  1.6984e+00],
        [ 1.3046e+01,  1.6192e+00,  8.0633e-01],
        [ 1.2827e+01,  1.1426e+00,  1.6932e+00],
        [ 1.2495e+01,  1.5423e+00,  1.1288e+00],
        [ 1.2178e+01,  9.7060e-01,  1.3755e+00],
        [ 1.1739e+01,  1.6789e+00,  1.9266e+00],
        [ 1.1173e+01,  1.6419e+00,  2.2388e+00],
        [ 1.0764e+01,  1.9675e+00,  1.7235e+00],
        [ 1.0623e+01,  1.6883e+00,  1.9361e+00],
        [ 1.0008e+01,  1.4814e+00,  8.3415e-01],
        [ 9.7410e+00,  1.6472e+00,  1.8286e+00],
        [ 9.0136e+00,  9.0139e-01,  1.3402e+00],
        [ 8.1371e+00,  1.6997e+00,  1.7721e+00],
        [ 6.8398e+00,  7.5818e-01,  1.5878e+00],
        [ 5.6292e+00

In [11]:
# Pocket Skeleton Generation algorithm prototyping

In [12]:
from src.nano_maker.modules.nAAno.radialseeker import RadialSeeker

In [13]:
r_mod = RadialSeeker(100, 33, False)

In [14]:
def process_skeleton(pocket_skeleton):
    output = []
    for vector in pocket_skeleton:
        output.append(vector.detach().flatten().tolist())
    return output

In [15]:
processed_sample = process_skeleton(sample_skeleton)
processed_sample

[[16.567577362060547, 1.731879472732544, 1.5057306289672852],
 [15.356108665466309, 1.5183837413787842, 1.5981448888778687],
 [14.992478370666504, 1.6651099920272827, 1.5024570226669312],
 [14.330421447753906, 1.220508098602295, 1.2933059930801392],
 [14.02639389038086, 1.3904579877853394, 1.9733084440231323],
 [13.602289199829102, 1.3843971490859985, 0.9934849739074707],
 [13.414308547973633, 2.0995006561279297, 1.6984354257583618],
 [13.045809745788574, 1.6192184686660767, 0.8063330054283142],
 [12.827056884765625, 1.1426005363464355, 1.693246841430664],
 [12.494571685791016, 1.5422838926315308, 1.1287944316864014],
 [12.177638053894043, 0.9705966114997864, 1.3754520416259766],
 [11.739079475402832, 1.67891263961792, 1.926648497581482],
 [11.173385620117188, 1.641889214515686, 2.2388486862182617],
 [10.76385498046875, 1.9674643278121948, 1.7235119342803955],
 [10.622983932495117, 1.6883054971694946, 1.936142921447754],
 [10.007525444030762, 1.4814364910125732, 0.8341473937034607],
 [

In [16]:
true_sample = []
for vector in processed_sample:
    appending = True
    if abs(vector[0]) < 0.33:
        break
    true_sample.append(vector)

true_sample

[[16.567577362060547, 1.731879472732544, 1.5057306289672852],
 [15.356108665466309, 1.5183837413787842, 1.5981448888778687],
 [14.992478370666504, 1.6651099920272827, 1.5024570226669312],
 [14.330421447753906, 1.220508098602295, 1.2933059930801392],
 [14.02639389038086, 1.3904579877853394, 1.9733084440231323],
 [13.602289199829102, 1.3843971490859985, 0.9934849739074707],
 [13.414308547973633, 2.0995006561279297, 1.6984354257583618],
 [13.045809745788574, 1.6192184686660767, 0.8063330054283142],
 [12.827056884765625, 1.1426005363464355, 1.693246841430664],
 [12.494571685791016, 1.5422838926315308, 1.1287944316864014],
 [12.177638053894043, 0.9705966114997864, 1.3754520416259766],
 [11.739079475402832, 1.67891263961792, 1.926648497581482],
 [11.173385620117188, 1.641889214515686, 2.2388486862182617],
 [10.76385498046875, 1.9674643278121948, 1.7235119342803955],
 [10.622983932495117, 1.6883054971694946, 1.936142921447754],
 [10.007525444030762, 1.4814364910125732, 0.8341473937034607],
 [

In [17]:
# convert everything into raw angstrom coordinates
angstrom_output = []
for vector in true_sample:
    angstrom_output.append(r_mod.radial_to_xyz(vector))
angstrom_output

[[-2.6516082882748258, 16.31849224234475, 1.0772205298580129],
 [0.8041840723744551, 15.32928676089, -0.419915141131254],
 [-1.4086045657844786, 14.891007992394695, 1.023778219435952],
 [4.7296261596415485, 12.945283504440061, 3.9257165269478578],
 [2.314744343146786, 12.696115943704015, -5.494572243406046],
 [2.112260632651174, 11.200374117035395, 7.423762262760245],
 [-6.711335243378071, 11.488513538250997, -1.7075449489408565],
 [-0.4557581685089514, 9.404827817298859, 9.029653932577432],
 [5.286300032909107, 11.581610045030255, -1.5667574962107862],
 [0.3219703258960847, 11.28921753138399, 5.344550893171968],
 [6.747205075762335, 9.858136975948112, 2.3637317108123606],
 [-1.1873550619065378, 10.939376643619175, -4.089769374405872],
 [-0.6230617610684758, 8.74928188324978, -6.921445426005535],
 [-4.110186153910965, 9.812531679095331, -1.637426599121644],
 [-1.1632289692410933, 9.853443080145574, -3.7953057138390083],
 [0.6615266369100733, 7.383237557675364, 6.723150391632888],
 [-0.

In [18]:
print(len(angstrom_output))

22


In [19]:
model.eval()
with torch.no_grad():
    coord_context = torch.tensor([[33, 0, 0] for _ in range(75)]).unsqueeze(0)


    smiles_list = [
        ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
        ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
        ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ]

    for name, smi in smiles_list:
        fp = torch.tensor(smiles_fingerprint(smi),
                         dtype=torch.float32).unsqueeze(0).to(device)
        out, _ = model(coord_context, fp)
        print(f"{name}: {out.squeeze().tolist()}")

Aspirin: [16.49224090576172, 0.07140648365020752, 1.5995190143585205]
Caffeine: [18.1778507232666, 1.0541880130767822, 1.3329460620880127]
Ibuprofen: [17.816110610961914, 1.152420163154602, 1.2482268810272217]


In [ ]:
# would be good to track n_trainable parameters